# Phase 4 — Hyperparameter Sweep

**Optional phase.** Skip to Phase 5 if your Phase 3 metrics already hit the threshold in README.

**Critical:** The `primary_metric` name must match the string you pass to `mlflow.log_metric()` in `train.py` exactly — including case. If they differ, the sweep cannot rank trials.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PROJECT_NAME    = 'my-project'

DATA_ASSET_NAME = f'{PROJECT_NAME}-data'
COMPUTE_NAME    = 'aml-cluster'
ENV_NAME        = f'{PROJECT_NAME}-env'
EXPERIMENT_NAME = f'{PROJECT_NAME}-training'

# Must match mlflow.log_metric() key in train.py exactly
PRIMARY_METRIC  = 'val_auc'
GOAL            = 'Maximize'   # 'Maximize' or 'Minimize'

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
from azure.ai.ml import MLClient, Input, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.sweep import Choice, Uniform
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

## Section 1 — Connect

In [ ]:
# ── AUTH ──────────────────────────────────────────────────────────────────────
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(f'✓ Connected to: {ml_client.workspace_name}')

## Section 2 — Define Search Space

In [ ]:
# ── BASE JOB ──────────────────────────────────────────────────────────────────
# Same as Phase 3 but the hyperparameter becomes a sweep expression.
job = command(
    code='../src',
    command='python train.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}',
    inputs={
        'training_data': Input(
            type=AssetTypes.URI_FILE,
            path=f'azureml:{DATA_ASSET_NAME}@latest',
        ),
        'reg_rate': 0.01,   # placeholder; overridden by sweep
    },
    environment=f'{ENV_NAME}@latest',
    compute=COMPUTE_NAME,
)

In [ ]:
# ── SWEEP JOB ─────────────────────────────────────────────────────────────────
# Choice → discrete values  |  Uniform → continuous range
sweep_job = job.sweep(
    sampling_algorithm='grid',
    primary_metric=PRIMARY_METRIC,
    goal=GOAL,
)

sweep_job.search_space = {
    'reg_rate': Choice(values=[0.001, 0.01, 0.1, 1.0]),
    # 'max_depth': Choice(values=[3, 5, 10]),            # example for tree models
    # 'learning_rate': Uniform(min_value=0.01, max_value=0.3),
}

sweep_job.set_limits(
    max_total_trials=4,
    max_concurrent_trials=2,
    timeout=7200,       # seconds — 2 hours hard limit
)

sweep_job.experiment_name = EXPERIMENT_NAME
sweep_job.display_name    = 'sweep-v1'

print('✓ Sweep job defined')

## Section 3 — Submit

In [ ]:
# ── SUBMIT SWEEP ──────────────────────────────────────────────────────────────
returned_sweep = ml_client.jobs.create_or_update(sweep_job)

print(f'✓ Sweep submitted: {returned_sweep.name}')
print(f'  Status         : {returned_sweep.status}')
print(f'  Studio URL     : {returned_sweep.studio_url}')

In [ ]:
# ── WAIT ──────────────────────────────────────────────────────────────────────
ml_client.jobs.stream(returned_sweep.name)

## Section 4 — Identify Best Trial

In [ ]:
# ── BEST TRIAL ────────────────────────────────────────────────────────────────
# In Studio: Jobs → [sweep job name] → Trials tab → sort by val_auc.
# The best child job name is needed in Phase 5 to register the model.
finished_sweep = ml_client.jobs.get(returned_sweep.name)

print(f'Sweep status: {finished_sweep.status}')
print()
print('→ Open Studio URL above')
print('→ Trials tab → sort by val_auc descending')
print('→ Copy the best CHILD job name (not the sweep name)')
print('→ Paste it into Phase 5 as BEST_JOB_NAME')

---
## Phase 4 Gate

- [ ] All trials status = `Completed`
- [ ] Best hyperparameters recorded in README → Hyperparameters table
- [ ] Best child job name saved (needed for Phase 5)